In [1]:
#Import packages 

import numpy as np 
import geopandas as gpd 
import matplotlib.pyplot as plt 
# from matplotlib.colors import ListedColormap
import matplotlib.colors as mcolors
import pandas as pd 
from shapely.geometry import shape 
import json 
from shapely import wkt 
from shapely.geometry import Point
from shapely.geometry import box
from math import cos, radians
from matplotlib.colors import Normalize
from mpl_toolkits.axes_grid1 import make_axes_locatable
from matplotlib.cm import ScalarMappable
import seaborn as sns 
import matplotlib
from statsmodels.tsa.seasonal import seasonal_decompose

import glob
import os
import csv
import ast

from scipy.stats import chi2_contingency
from math import sqrt
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
from pathlib import Path
import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning

In [2]:
DATA_DIR  = Path('../files')
PLOTS_DIR = Path('../outputs/plots')
PLOTS_DIR.mkdir(parents=True, exist_ok=True)
(PLOTS_DIR / 'supplementary').mkdir(exist_ok=True)
(PLOTS_DIR / 'spatial').mkdir(exist_ok=True)

In [3]:
warnings.filterwarnings("ignore", category=ConvergenceWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

### Reading Temp & Precip (6km) 

In [4]:
temp_all = pd.read_csv(DATA_DIR / 'processed_weather_data/temp_all_years_6km_buffer.csv').drop(columns = ['Unnamed: 0'])
temp_all['time'] = pd.to_datetime(temp_all['time'])

precip_all = pd.read_csv(DATA_DIR / 'processed_weather_data/precip_all_years_6km_buffer.csv').drop(columns = ['Unnamed: 0'])
precip_all['time'] = pd.to_datetime(precip_all['time'])

### Wind (6km) 

In [5]:
hourly_wind_df = pd.read_csv(DATA_DIR / 'processed_weather_data/ea_hourly_wind_avg_6km.csv').drop(columns=['Unnamed: 0'])
hourly_wind_df['Timestamp'] = pd.to_datetime(hourly_wind_df['Timestamp'])
hourly_wind_df = hourly_wind_df.rename(columns = {'Timestamp':'time'})
hourly_wind_df['Date'] = hourly_wind_df['time'].dt.date

### Lightning (6km) 

In [6]:
hourly_lightning_df = pd.read_csv(DATA_DIR / 'processed_weather_data/ea_hourly_lightning_avg_6km_buffer_aligned.csv').drop(columns=['Unnamed: 0'])
hourly_lightning_df['Timestamp'] = pd.to_datetime(hourly_lightning_df['Timestamp'])
hourly_lightning_df = hourly_lightning_df.rename(columns = {'Timestamp':'time'})
hourly_lightning_df['Date'] = hourly_lightning_df['time'].dt.date

### Extreme Definitions 

### Temp 

In [7]:
# temp_all['Temp'].quantile(0.95)

In [8]:
temp_all['Date'] = temp_all['time'].dt.floor('D')

# Hot hour flag
temp_all['Hot_Hour'] = temp_all['Temp'] > 32

# Group by EA and Date, and count Hot_Hour sum
daily_hot_hours = (
    temp_all.groupby(['ea_code9ch', 'Date'], as_index=False)
            .agg({'Hot_Hour': 'sum'})
)

daily_hot_hours = daily_hot_hours.rename(columns={'Hot_Hour': 'Num_Hot_Hours'})

### Precip 

In [9]:
precip_all['Date'] = precip_all['time'].dt.floor('D')  

# Group by EA and Date, sum Precip
daily_precip = (
    precip_all.groupby(['ea_code9ch', 'Date'], as_index=False)
      .agg({'Precip': 'sum'})
)

### Wind 

In [10]:
# Windy hour flag
hourly_wind_df['Windy_Hour'] = hourly_wind_df['Wind Gusts (m/s)'] > 5.93

# Group by EA and Date, and count Windy_Hour sum
daily_windy_hours = (
    hourly_wind_df.groupby(['ea_code9ch', 'Date'], as_index=False)
            .agg({'Windy_Hour': 'sum'})
)

daily_windy_hours = daily_windy_hours.rename(columns={'Windy_Hour': 'Num_Windy_Hours'})

### Lightning 

In [11]:
daily_lightning_per_ea = hourly_lightning_df.groupby(['ea_code9ch', 'Date'])['Lightning Events'].sum().reset_index()

## Percentiles 

In [12]:
### 90th pct 
hot_hrs_90_thresh = daily_hot_hours['Num_Hot_Hours'].quantile(0.90)
temp_90_thresh = temp_all['Temp'].quantile(0.90)
precip_90_thresh = daily_precip['Precip'].quantile(0.90)
lightning_90_thresh = daily_lightning_per_ea['Lightning Events'].quantile(0.90)
windy_hrs_90_thresh = daily_windy_hours['Num_Windy_Hours'].quantile(0.90)
wind_90_thresh = hourly_wind_df['Wind Gusts (m/s)'].quantile(0.90)

### 95th pct 
hot_hrs_95_thresh = daily_hot_hours['Num_Hot_Hours'].quantile(0.95)
temp_95_thresh = temp_all['Temp'].quantile(0.95)
precip_95_thresh = daily_precip['Precip'].quantile(0.95)
lightning_95_thresh = daily_lightning_per_ea['Lightning Events'].quantile(0.95)
windy_hrs_95_thresh = daily_windy_hours['Num_Windy_Hours'].quantile(0.95)
wind_95_thresh = hourly_wind_df['Wind Gusts (m/s)'].quantile(0.95)

### 99th pct 
hot_hrs_99_thresh = daily_hot_hours['Num_Hot_Hours'].quantile(0.99)
temp_99_thresh = temp_all['Temp'].quantile(0.99)
precip_99_thresh = daily_precip['Precip'].quantile(0.99)
lightning_99_thresh = daily_lightning_per_ea['Lightning Events'].quantile(0.99)
windy_hrs_99_thresh = daily_windy_hours['Num_Windy_Hours'].quantile(0.99)
wind_99_thresh = hourly_wind_df['Wind Gusts (m/s)'].quantile(0.99)

#### Dictionary for percentiles 

In [13]:
# Percentile Threshold dictionaries 
temp_thresh_dict = {
    '90': temp_90_thresh.round(2),
    '95': temp_95_thresh.round(2),
    '99': temp_99_thresh.round(2)
}

hot_hrs_thresh_dict = {
    '90': hot_hrs_90_thresh.round(2),
    '95': hot_hrs_95_thresh.round(2),
    '99': hot_hrs_99_thresh.round(2)
}

precip_thresh_dict = {
    '90': precip_90_thresh.round(2),
    '95': precip_95_thresh.round(2),
    '99': precip_99_thresh.round(2)
}

wind_thresh_dict = {
    '90': wind_90_thresh.round(2),
    '95': wind_95_thresh.round(2),
    '99': wind_99_thresh.round(2)
}

windy_hrs_thresh_dict = {
    '90': windy_hrs_90_thresh.round(2),
    '95': windy_hrs_95_thresh.round(2),
    '99': windy_hrs_99_thresh.round(2)
}

lightning_thresh_dict = {
    '90': 1,   # at least 1 lightning strike 
    '95': lightning_95_thresh.round(2),
    '99': lightning_99_thresh.round(2)
}

### Read EAs and CESI level 

In [14]:
eas_n_cesi_level = pd.read_csv(DATA_DIR / 'miscellaneous/eas_cesi_level_214_eas.csv')

### EA Names mapped  

In [15]:
ea_names_mapped = pd.read_csv(DATA_DIR / 'miscellaneous/ea_names_mapped.csv')

ea_names_mapped = ea_names_mapped[['ea_code9ch', 'LOC_NAME', 'BASE_NAM', 'geometry']]

### EAs n Sites  (6km) 

In [16]:
merged_eas_sites = pd.read_csv(DATA_DIR / 'miscellaneous/ea_site_list_6km_buffer.csv')
merged_eas_sites = merged_eas_sites[['ea_code9ch', 'Intersecting_Sites']]

# Convert the string representation of lists to actual lists
merged_eas_sites['Intersecting_Sites'] = merged_eas_sites['Intersecting_Sites'].apply(ast.literal_eval)

In [17]:
# merged_eas_sites['ea_code9ch'].nunique()

In [18]:
filtered_eas_sites_copy = merged_eas_sites

In [19]:
# EA -> 30200014, 30500020 

In [20]:
def prepare_hourly_weather_df_TPLW(
    ea_row, 
    temp_df, 
    precip_df, 
    lightning_df, 
    wind_df, 
):
    ea = ea_row['ea_code9ch']
    site_list = [int(s) for s in ea_row['Intersecting_Sites']]
    all_sites_dfs = []

    for site_id in site_list:
        # --- Temperature ---
        temp_filt = (
            temp_df[temp_df['ea_code9ch'] == ea]
            .drop(columns=['geometry'], errors='ignore')
            .copy()
        )
        if 'Hot_Hour' in temp_filt.columns:
            temp_filt = temp_filt.drop(columns=['Hot_Hour'])
        temp_filt['time'] = pd.to_datetime(temp_filt['time'])

        # --- Precipitation ---
        precip_filt = (
            precip_df[precip_df['ea_code9ch'] == ea]
            .drop(columns=['geometry'], errors='ignore')
            [['time', 'Precip']]
        )
        precip_filt['time'] = pd.to_datetime(precip_filt['time'])

        # Merge temp + precip
        merged = temp_filt.merge(precip_filt, on='time', how='outer')

        # --- Lightning ---
        lightning_filt = (
            lightning_df[lightning_df['ea_code9ch'] == ea]
            .drop(columns=['geometry'], errors='ignore')
            [['time', 'Lightning Events']]
        )
        merged = merged.merge(lightning_filt, on='time', how='outer')
        merged['Lightning Events'] = merged['Lightning Events'].fillna(0)

        # --- Wind ---
        wind_filt = (
            wind_df[wind_df['ea_code9ch'] == ea]
            .drop(columns=['geometry'], errors='ignore')
            [['time', 'Wind Gusts (m/s)']]
        )
        merged = merged.merge(wind_filt, on='time', how='inner')

        # Add site_id
        merged['site_id'] = site_id

        all_sites_dfs.append(merged)

    # Concatenate all sites
    df_all = pd.concat(all_sites_dfs, ignore_index=True)

    # Aggregate by timestamp 
    agg_dict = {
        'Temp': 'mean',  # average across sites
        'Precip': 'mean',
        'Lightning Events': 'mean',
        'Wind Gusts (m/s)': 'mean',
        'site_id': lambda x: ','.join(map(str, x))  # list all contributing sites
    }

    df_agg = df_all.groupby('time', as_index=False).agg(agg_dict)

    # Add EA code
    df_agg['ea_code9ch'] = ea

    # Optional: reorder columns
    df_agg = df_agg[
        ['time', 'ea_code9ch', 'site_id', 'Temp', 'Precip', 
         'Lightning Events', 'Wind Gusts (m/s)']
    ]

    return df_agg

## Remove sites with > `10%` missing data & less than 24 months 

In [21]:
sites_to_omit = pd.read_csv(DATA_DIR / 'miscellaneous/complete_site_removal_df.csv')['site_id'].to_list()

### Resample hourly weather data -> Daily 

In [22]:
filtered_eas_sites_copy_r1 = merged_eas_sites.copy()

In [23]:
# Main loop
results = []

for _, row in filtered_eas_sites_copy_r1.iterrows():
    merged_hourly = prepare_hourly_weather_df_TPLW(
        row, 
        temp_all, 
        precip_all, 
        hourly_lightning_df, 
        hourly_wind_df, 
    )
    if merged_hourly is not None:
        results.append(merged_hourly)

merged_hourly_data_global = pd.concat(results, ignore_index=True)

In [24]:
merged_hourly_data_global = merged_hourly_data_global.loc[
    ~merged_hourly_data_global['site_id'].isin(sites_to_omit)
]

In [25]:
### Filter 214 EAs 

tplw_eas = pd.read_csv(DATA_DIR / 'miscellaneous/unique_ea_codes_TPLW.csv')

list_EAs = tplw_eas['ea_code9ch'].unique().tolist()

hourly_global_test = merged_hourly_data_global[merged_hourly_data_global['ea_code9ch'].isin(list_EAs)].reset_index(drop=True)

In [26]:
hourly_global_test['ea_code9ch'].nunique()

214

In [27]:
# hourly_global_test

In [28]:
# 1. Convert to string (just in case)
hourly_global_test['site_id'] = hourly_global_test['site_id'].astype(str)

# 2. Split into list
hourly_global_test['site_id'] = hourly_global_test['site_id'].str.split(',')

# 3. Explode into separate rows
hourly_global_test = hourly_global_test.explode('site_id')

# 4. Clean up (remove spaces + convert to int)
hourly_global_test['site_id'] = hourly_global_test['site_id'].str.strip().astype(int)

### Select percentile, lag, saidi_threshold, agg_method 

In [29]:
percentile = '95'
selected_lags = [0]
saidi_threshold = 0
agg_method = 'mean'

## Daily Resampling 

### Temp 

In [30]:
def compute_hot_flags_daily(
    df,
    temp_thresh=31.257,
    min_hot_hours=4
):
    import pandas as pd

    df = df.copy()

    # Ensure datetime
    df['Date'] = df['time'].dt.date

    # Hot hour flag
    df['Hot_hour'] = df['Temp'] >= temp_thresh

    # Aggregate to daily
    daily_df = (
        df.groupby(['ea_code9ch', 'site_id', 'Date'])
          .agg(
              Hot_hour_count=('Hot_hour', 'sum'),
              mean_temp=('Temp', 'mean'),
              max_temp=('Temp', 'max')
          )
          .reset_index()
    )

    # Hot day flag
    daily_df['Hot_day'] = daily_df['Hot_hour_count'] >= min_hot_hours

    return daily_df

In [31]:
processed_temp = compute_hot_flags_daily(
    
    hourly_global_test,
    
    temp_thresh = temp_thresh_dict[percentile],
    min_hot_hours = hot_hrs_thresh_dict[percentile],    

)

### PWL 

In [32]:
def compute_pwl_flags_daily(
    df,
    precip_thresh=10.0,
    lightning_thresh=5,
    wind_thresh=None,
    min_wind_hours=4
):
    df = df.copy()

    # Date
    df['Date'] = df['time'].dt.date

    # ---------------- WIND ----------------
    df['Wind_hour'] = df['Wind Gusts (m/s)'] >= wind_thresh

    wind_daily = (
        df.groupby(['ea_code9ch', 'site_id', 'Date'])
          .agg(Wind_hour_count=('Wind_hour', 'sum'))
          .reset_index()
    )
    wind_daily['Wind_day'] = wind_daily['Wind_hour_count'] >= min_wind_hours

    # ---------------- PRECIP ----------------
    precip_daily = (
        df.groupby(['ea_code9ch', 'site_id', 'Date'])
          .agg(Precip_sum=('Precip', 'sum'))
          .reset_index()
    )
    precip_daily['Precip_day'] = precip_daily['Precip_sum'] >= precip_thresh

    # ---------------- LIGHTNING ----------------
    lightning_daily = (
        df.groupby(['ea_code9ch', 'site_id', 'Date'])
          .agg(Lightning_sum=('Lightning Events', 'sum'))
          .reset_index()
    )
    lightning_daily['Lightning_day'] = lightning_daily['Lightning_sum'] >= lightning_thresh

    # ---------------- MERGE ALL ----------------
    daily_df = wind_daily.merge(
        precip_daily, on=['ea_code9ch', 'site_id', 'Date'], how='outer'
    ).merge(
        lightning_daily, on=['ea_code9ch', 'site_id', 'Date'], how='outer'
    )

    # Fill NaNs
    daily_df[['Wind_day','Precip_day','Lightning_day']] = (
        daily_df[['Wind_day','Precip_day','Lightning_day']].fillna(False)
    )

    # ---------------- COMBINED EVENTS ----------------
    daily_df['PL_day'] = daily_df['Precip_day'] & daily_df['Lightning_day']

    daily_df['PWL_day'] = (
        daily_df['Precip_day'] &
        daily_df['Wind_day'] &
        daily_df['Lightning_day']
    )

    return daily_df

In [33]:
processed_pwl = compute_pwl_flags_daily(
    
    hourly_global_test,
    
    precip_thresh = precip_thresh_dict[percentile],           # Min total precipitation in a day for Precip Day
    lightning_thresh = lightning_thresh_dict[percentile],   
    
    wind_thresh = wind_thresh_dict[percentile],
    min_wind_hours = windy_hrs_thresh_dict[percentile],    
)

### Combined Weather Df  

In [34]:
combined_daily_weather = processed_pwl.merge(processed_temp, on = ['ea_code9ch', 'site_id', 'Date'])

combined_daily_weather = combined_daily_weather[['ea_code9ch', 'site_id', 'Date', 
                        'Hot_hour_count', 'Hot_day', 
                        'Wind_hour_count', 'Wind_day',
                        'Precip_sum', 'Precip_day', 
                        'Lightning_sum', 'Lightning_day', 
                        'PL_day', 'PWL_day']]

combined_daily_weather['Date'] = pd.to_datetime(combined_daily_weather['Date'])

In [35]:
combined_daily_weather.head()

,ea_code9ch,site_id,Date,Hot_hour_count,Hot_day,Wind_hour_count,Wind_day,Precip_sum,Precip_day,Lightning_sum,Lightning_day,PL_day,PWL_day
0,30200002,456,2022-01-01,0,False,0,False,0.000000,False,0.0,False,False,False
1,30200002,456,2022-01-02,0,False,0,False,4.477796,False,0.0,False,False,False
2,30200002,456,2022-01-03,3,False,0,False,0.162920,False,0.0,False,False,False
3,30200002,456,2022-01-04,0,False,0,False,0.041221,False,0.0,False,False,False
4,30200002,456,2022-01-05,0,False,0,False,0.016973,False,0.0,False,False,False


## read SAIDI data  

In [36]:
# read data 
daily_saidi = pd.read_csv(DATA_DIR / 'SAIDI_SAIFI/SAIDI_Daily_Site Id - 31_03_2016 to 31_03_2026.csv')

# Convert from milliseconds to datetime
daily_saidi['Date'] = pd.to_datetime(daily_saidi['time'], unit='ms')

# Extract year
daily_saidi['year'] = daily_saidi['Date'].dt.year

# Filter for 2022 and 2023
daily_saidi = daily_saidi[daily_saidi['year'].isin([2022, 2023])].reset_index(drop=True)

### Prep data for Regression 

In [37]:
combined_saidi_weather = combined_daily_weather.merge(daily_saidi, on = ['Date', 'site_id'])

combined_saidi_weather = combined_saidi_weather[['ea_code9ch', 'site_id', 'Date', 'SAIDI', 
                        'Hot_hour_count', 'Hot_day',
                        'Wind_hour_count', 'Wind_day', 
                        'Precip_sum', 'Precip_day',
                       'Lightning_sum', 'Lightning_day', 
                        'PL_day', 'PWL_day']]

## add Month and Season 
combined_saidi_weather['Month'] = combined_saidi_weather['Date'].dt.month
combined_saidi_weather['season'] = np.where(combined_saidi_weather['Month'].isin([12, 1, 2, 3]),'dry', np.where(combined_saidi_weather['Month'] == 8, 'transition', 'rainy'))

In [38]:
### rename column names 

In [39]:
combined_saidi_weather = combined_saidi_weather.rename(columns = {
    'Hot_day':'T', 
    'Precip_day':'P', 
    'Lightning_day':'L', 
    'Wind_day':'W'
})

In [40]:
combined_saidi_weather.head()

,ea_code9ch,site_id,Date,SAIDI,Hot_hour_count,T,Wind_hour_count,W,Precip_sum,P,Lightning_sum,L,PL_day,PWL_day,Month,season
0,30200002,456,2022-01-01,0.0,0,False,0,False,0.000000,False,0.0,False,False,False,1,dry
1,30200002,456,2022-01-02,0.0,0,False,0,False,4.477796,False,0.0,False,False,False,1,dry
2,30200002,456,2022-01-03,0.0,3,False,0,False,0.162920,False,0.0,False,False,False,1,dry
3,30200002,456,2022-01-04,0.0,0,False,0,False,0.041221,False,0.0,False,False,False,1,dry
4,30200002,456,2022-01-05,0.0,0,False,0,False,0.016973,False,0.0,False,False,False,1,dry


### merge CESI to the df 

In [41]:
cesi_eas_df = pd.read_csv(DATA_DIR / 'miscellaneous/eas_cesi_level_214_eas_rev1.csv').drop(columns = ['Unnamed: 0'])

combined_saidi_weather = combined_saidi_weather.merge(cesi_eas_df, on = 'ea_code9ch', how = 'left')

In [42]:
combined_saidi_weather.head()

,ea_code9ch,site_id,Date,SAIDI,Hot_hour_count,T,Wind_hour_count,W,Precip_sum,P,Lightning_sum,L,PL_day,PWL_day,Month,season,cesi_level
0,30200002,456,2022-01-01,0.0,0,False,0,False,0.000000,False,0.0,False,False,False,1,dry,Low
1,30200002,456,2022-01-02,0.0,0,False,0,False,4.477796,False,0.0,False,False,False,1,dry,Low
2,30200002,456,2022-01-03,0.0,3,False,0,False,0.162920,False,0.0,False,False,False,1,dry,Low
3,30200002,456,2022-01-04,0.0,0,False,0,False,0.041221,False,0.0,False,False,False,1,dry,Low
4,30200002,456,2022-01-05,0.0,0,False,0,False,0.016973,False,0.0,False,False,False,1,dry,Low


### Aggregate combined_saidi_weather per EA first 

In [43]:
import numpy as np
import pandas as pd

def aggregate_saidi_to_ea(
    df,
    ea_col='ea_code9ch',
    site_col='site_id',
    date_col='Date',
    saidi_col='SAIDI',
    saidi_agg='mean'
):
    df = df.copy()
    df[date_col] = pd.to_datetime(df[date_col])

    valid_aggs = ['mean', 'median', 'max']
    if saidi_agg not in valid_aggs:
        raise ValueError(f"saidi_agg must be one of {valid_aggs}")

    weather_cols = [
        'Hot_hour_count', 'T',
        'Wind_hour_count', 'W',
        'Precip_sum', 'P',
        'Lightning_sum', 'L',
        'PL_day', 'PWL_day',
        'Month', 'season',
        'cesi_level'  # added here
    ]

    weather_cols = [c for c in weather_cols if c in df.columns]

    agg_dict = {saidi_col: saidi_agg}
    agg_dict.update({col: 'first' for col in weather_cols})

    site_counts = (
        df.groupby([ea_col, date_col])[site_col]
        .nunique()
        .rename('n_sites')
        .reset_index()
    )

    df_ea = (
        df.groupby([ea_col, date_col], as_index=False)
        .agg(agg_dict)
        .merge(site_counts, on=[ea_col, date_col], how='left')
    )

    return df_ea

## *** Finding the EAs with significant EW <-> SAIDI relationship 

### Functions 

#### diagnostics function 

In [44]:
### new col names 

In [45]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from itertools import combinations

def diagnose_separation_issues(
    df,
    lags,
    base_cols=None,
    id_col='ea_code9ch',
    date_col='Date',
    outcome_col='SAIDI',
    saidi_threshold=0,
    control_months=False,
    coef_threshold=10
):
    df_model = df.copy()
    df_model[date_col] = pd.to_datetime(df_model[date_col])
    df_model = df_model.sort_values([id_col, date_col])

    # Create binary outage variable
    if saidi_threshold > 0:
        df_model['outage'] = (df_model[outcome_col] >= saidi_threshold).astype(int)
    else:
        df_model['outage'] = (df_model[outcome_col] > saidi_threshold).astype(int)

    # Default base cols
    if base_cols is None:
        base_cols = ['T', 'P', 'L', 'W']

    df_model[base_cols] = df_model[base_cols].astype(int)

    # Construct interactions
    df_model['PW']  = df_model['P'] * df_model['W']
    df_model['PL']  = df_model['P'] * df_model['L']
    df_model['WL']  = df_model['W'] * df_model['L']
    df_model['PWL'] = df_model['P'] * df_model['W'] * df_model['L']

    interaction_cols = ['PW', 'PL', 'WL', 'PWL']
    all_cols         = base_cols + interaction_cols

    results = []

    # ==========================================================
    # HELPER: Fit model safely
    # ==========================================================
    def fit_logit(X, y):
        try:
            model = sm.Logit(y, X).fit(disp=0)
            issues = []

            if not model.mle_retvals['converged']:
                issues.append('non_converged')

            if np.any(np.abs(model.params) > coef_threshold):
                issues.append('large_coef')

            if not np.isfinite(model.llf):
                issues.append('invalid_loglik')

            return model, issues

        except Exception:
            return None, ['fit_failed']

    # ==========================================================
    # LOOP OVER EAs
    # ==========================================================
    for ea, df_ea in df_model.groupby(id_col):

        for lag in lags:
            temp_df = df_ea.copy()
            ew_cols = []

            # Build lagged EW variables (main effects + interactions)
            for col in all_cols:
                ew_col = f'EW_{col}_{lag}day'
                if lag == 0:
                    temp_df[ew_col] = temp_df[col]
                else:
                    temp_df[ew_col] = temp_df[col].rolling(window=lag + 1, min_periods=1).max().astype(int)
                ew_cols.append(ew_col)

            # Drop zero-variance columns
            valid_ew_cols = [c for c in ew_cols if temp_df[c].nunique() > 1]
            dropped_zero  = [c for c in ew_cols if c not in valid_ew_cols]

            # Build controls
            if control_months:
                month_dummies = pd.get_dummies(
                    temp_df[date_col].dt.month, prefix='month', drop_first=True
                ).astype(int)
                month_dummies.index = temp_df.index
                X_full = pd.concat([temp_df[valid_ew_cols], month_dummies], axis=1)
            else:
                month_dummies = None
                X_full = temp_df[valid_ew_cols]

            X_full = sm.add_constant(X_full)
            y = temp_df['outage']

            # Skip if no variation in outcome
            if y.nunique() < 2:
                results.append({
                    'ea_code9ch':             ea,
                    'lag':            lag,
                    'status':         'no_variation',
                    'initial_issues': [],
                    'culprit_vars':   [],
                    'dropped_zero':   dropped_zero,
                    'used_vars':      [],
                    'stable_vars':    None
                })
                continue

            # ======================================================
            # 1. FIT FULL MODEL
            # ======================================================
            model, issues = fit_logit(X_full, y)

            if not issues:
                results.append({
                    'ea_code9ch':     ea,
                    'lag':            lag,
                    'status':         'ok',
                    'initial_issues': [],
                    'culprit_vars':   [],
                    'dropped_zero':   dropped_zero,
                    'used_vars':      valid_ew_cols,
                    'stable_vars':    valid_ew_cols.copy()
                })
                continue

            # ======================================================
            # 2. IDENTIFY CULPRIT VARIABLES (INDIVIDUAL TEST)
            # ======================================================
            culprit_vars = []

            for var in valid_ew_cols:
                X_test = sm.add_constant(temp_df[[var]])
                _, var_issues = fit_logit(X_test, y)
                if var_issues:
                    culprit_vars.append(var)

            # ======================================================
            # 3. TRY DROPPING VARIABLES (STEPWISE)
            # ======================================================
            stable_vars = None

            for k in range(1, len(valid_ew_cols) + 1):
                for combo in combinations(valid_ew_cols, k):
                    remaining = [v for v in valid_ew_cols if v not in combo]

                    if not remaining:
                        continue

                    if control_months:
                        X_try = pd.concat([temp_df[remaining], month_dummies], axis=1)
                    else:
                        X_try = temp_df[remaining]

                    X_try = sm.add_constant(X_try)
                    _, try_issues = fit_logit(X_try, y)

                    if not try_issues:
                        stable_vars = remaining
                        break

                if stable_vars is not None:
                    break

            # ======================================================
            # 4. STORE RESULTS
            # ======================================================
            results.append({
                'ea_code9ch':     ea,
                'lag':            lag,
                'status':         'unstable',
                'initial_issues': issues,
                'culprit_vars':   culprit_vars,
                'dropped_zero':   dropped_zero,
                'used_vars':      valid_ew_cols,
                'stable_vars':    stable_vars
            })

    results_df = pd.DataFrame(results)

    expected_cols = [
        'ea_code9ch', 'lag', 'status', 'initial_issues',
        'culprit_vars', 'dropped_zero', 'used_vars', 'stable_vars'
    ]
    for col in expected_cols:
        if col not in results_df.columns:
            results_df[col] = None

    # ==========================================================
    # SUMMARY PER EA
    # ==========================================================
    if not results_df.empty:
        ea_summary = (
            results_df.groupby('ea_code9ch')
            .agg({
                'status':       lambda x: list(set(x)),
                'culprit_vars': lambda x: list(set(tuple(v) for v in x if isinstance(v, list) and len(v) > 0))
            })
            .reset_index()
        )
    else:
        ea_summary = pd.DataFrame()

    return results_df, ea_summary

#### regression table function 

In [46]:
### new column names 

In [47]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning


def run_ew_logit_with_diagnostics(
    df,
    diagnostics_df,
    lags,
    base_cols=None,
    id_col='ea_code9ch',
    date_col='Date',
    outcome_col='SAIDI',
    saidi_threshold=0,
    control_months=False,
    alpha=0.05
):
    warnings.filterwarnings("ignore", category=ConvergenceWarning)
    warnings.filterwarnings("ignore", category=RuntimeWarning)

    df_model = df.copy()
    df_model[date_col] = pd.to_datetime(df_model[date_col])
    df_model = df_model.sort_values([id_col, date_col])

    # Create binary outage variable
    if saidi_threshold > 0:
        df_model['outage'] = (df_model[outcome_col] >= saidi_threshold).astype(int)
    else:
        df_model['outage'] = (df_model[outcome_col] > saidi_threshold).astype(int)

    # Default base cols
    if base_cols is None:
        base_cols = ['T', 'P', 'L', 'W']

    df_model[base_cols] = df_model[base_cols].astype(int)

    # Construct interactions
    df_model['PW']  = df_model['P'] * df_model['W']
    df_model['PL']  = df_model['P'] * df_model['L']
    df_model['WL']  = df_model['W'] * df_model['L']
    df_model['PWL'] = df_model['P'] * df_model['W'] * df_model['L']

    interaction_cols = ['PW', 'PL', 'WL', 'PWL']
    all_cols         = base_cols + interaction_cols

    summary_rows = []
    coef_rows    = []

    # ==========================================================
    # LOOP OVER EAs
    # ==========================================================
    for ea, df_ea in df_model.groupby(id_col):

        for lag in lags:

            temp_df = df_ea.copy()
            ew_cols = []

            # Build lagged EW variables (main effects + interactions)
            for col in all_cols:
                ew_col = f'EW_{col}_{lag}day'
                if lag == 0:
                    temp_df[ew_col] = temp_df[col]
                else:
                    temp_df[ew_col] = temp_df[col].rolling(window=lag + 1, min_periods=1).max().astype(int)
                ew_cols.append(ew_col)

            # Drop zero-variance columns (consistent with diagnostics)
            valid_ew_cols = [c for c in ew_cols if temp_df[c].nunique() > 1]
            dropped_zero  = [c for c in ew_cols if c not in valid_ew_cols]

            # Build controls
            if control_months:
                month_dummies = pd.get_dummies(
                    temp_df[date_col].dt.month, prefix='month', drop_first=True
                ).astype(int)
                month_dummies.index = temp_df.index
            else:
                month_dummies = None

            y = temp_df['outage']

            # No outcome variation
            if y.nunique() < 2:
                summary_rows.append({
                    'ea_code9ch':     ea,
                    'lag':            lag,
                    'status':         'no_variation',
                    'model_type':     'none',
                    'log_likelihood': np.nan,
                    'llr_pvalue':     np.nan,
                    'significant_EW': [],
                    'dropped_zero':   dropped_zero,
                    'dropped_EW':     all_cols.copy()
                })
                continue

            # Pull diagnostics row
            diag_row = diagnostics_df[
                (diagnostics_df['ea_code9ch'] == ea) &
                (diagnostics_df['lag'] == lag)
            ]

            if not diag_row.empty:
                diag_row    = diag_row.iloc[0]
                status      = diag_row['status']
                stable_vars = diag_row['stable_vars']
            else:
                status      = 'unknown'
                stable_vars = None

            # Decide model specification
            if status == 'ok':
                model_type = 'full'
                use_vars   = valid_ew_cols

            elif status == 'unstable':
                if isinstance(stable_vars, list) and len(stable_vars) > 0:
                    model_type = 'reduced'
                    use_vars   = stable_vars
                else:
                    model_type = 'none'
                    use_vars   = None

            elif status == 'no_variation':
                model_type = 'none'
                use_vars   = None

            else:
                model_type = 'none'
                use_vars   = None

            # Determine dropped EW vars
            if model_type == 'full':
                dropped_EW = dropped_zero

            elif model_type == 'reduced':
                dropped_EW = [v for v in valid_ew_cols if v not in use_vars] + dropped_zero

            else:
                dropped_EW = all_cols.copy()

            # If no usable model
            if model_type == 'none' or use_vars is None or len(use_vars) == 0:
                summary_rows.append({
                    'ea_code9ch':     ea,
                    'lag':            lag,
                    'status':         status,
                    'model_type':     'none',
                    'log_likelihood': np.nan,
                    'llr_pvalue':     np.nan,
                    'significant_EW': [],
                    'dropped_zero':   dropped_zero,
                    'dropped_EW':     dropped_EW
                })
                continue

            # Build X
            if control_months:
                X = pd.concat([temp_df[use_vars], month_dummies], axis=1)
            else:
                X = temp_df[use_vars]
            X = sm.add_constant(X)

            # Fit model
            try:
                model = sm.Logit(y, X).fit(disp=0)
                if not model.mle_retvals['converged']:
                    raise ValueError("Non-converged model")

            except Exception:
                summary_rows.append({
                    'ea_code9ch':     ea,
                    'lag':            lag,
                    'status':         status,
                    'model_type':     'none',
                    'log_likelihood': np.nan,
                    'llr_pvalue':     np.nan,
                    'significant_EW': [],
                    'dropped_zero':   dropped_zero,
                    'dropped_EW':     dropped_EW
                })
                continue

            # Extract significant EW vars
            sig_ew_vars = []
            for col in all_cols:
                ew_name = f'EW_{col}_{lag}day'
                if ew_name in model.params.index:
                    pval = model.pvalues.get(ew_name, np.nan)
                    if pd.notna(pval) and pval < alpha:
                        sig_ew_vars.append(col)

            # Store summary row
            summary_rows.append({
                'ea_code9ch':     ea,
                'lag':            lag,
                'status':         status,
                'model_type':     model_type,
                'log_likelihood': round(model.llf, 3),
                'llr_pvalue':     model.llr_pvalue,
                'significant_EW': sig_ew_vars,
                'dropped_zero':   dropped_zero,
                'dropped_EW':     dropped_EW
            })

            # Store coefficient rows
            conf_int = model.conf_int()
            conf_int.columns = ['ci_lower', 'ci_upper']

            # Compute Average Marginal Effects (AME)
            try:
                marginal_effects = model.get_margeff(at='overall')
                me_frame = marginal_effects.summary_frame()
                me_frame.columns = ['me', 'me_se', 'me_z', 'me_pvalue', 'me_ci_lower', 'me_ci_upper']
            except Exception:
                me_frame = pd.DataFrame()

            for var in model.params.index:
                if var == 'const':
                    continue

                is_ew = var.startswith('EW_')

                # Pull marginal effect for this var if available
                me        = me_frame.loc[var, 'me']        if var in me_frame.index else np.nan
                me_se     = me_frame.loc[var, 'me_se']     if var in me_frame.index else np.nan
                me_pvalue = me_frame.loc[var, 'me_pvalue'] if var in me_frame.index else np.nan

                coef_rows.append({
                    'ea_code9ch':  ea,
                    'lag':         lag,
                    'status':      status,
                    'model_type':  model_type,
                    'variable':    var,
                    'is_EW':       is_ew,
                    'base_var': (
                        var.replace('EW_', '').replace(f'_{lag}day', '')
                        if is_ew else var
                    ),
                    'coef':        model.params[var],
                    'odds_ratio':  np.exp(model.params[var]),
                    'std_err':     model.bse[var],
                    'z_value':     model.tvalues[var],
                    'p_value':     model.pvalues[var],
                    'ci_lower':    conf_int.loc[var, 'ci_lower'],
                    'ci_upper':    conf_int.loc[var, 'ci_upper'],
                    'or_ci_lower': np.exp(conf_int.loc[var, 'ci_lower']),
                    'or_ci_upper': np.exp(conf_int.loc[var, 'ci_upper']),
                    'significant': pd.notna(model.pvalues[var]) and model.pvalues[var] < alpha,
                    # 'me':          me,
                    # 'me_se':       me_se,
                    # 'me_pvalue':   me_pvalue,
                    # 'me_sig':      pd.notna(me_pvalue) and me_pvalue < alpha,
                })

    results_df = pd.DataFrame(summary_rows)
    coef_df    = pd.DataFrame(coef_rows)

    return results_df, coef_df

### Functions for overall regression 

In [48]:
# use_clustered_se=True tells the model that all data is 'not entirely independent' and rows within the same EA may be correlated 

In [49]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from itertools import combinations


def diagnose_separation_overall(
    df,
    lags,
    base_cols=None,
    id_col='ea_code9ch',
    date_col='Date',
    outcome_col='SAIDI',
    saidi_threshold=0,
    control_months=False,
    coef_threshold=10,
    use_clustered_se=False,
    cluster_col='ea_code9ch'
):
    df_model = df.copy()
    df_model[date_col] = pd.to_datetime(df_model[date_col])
    df_model = df_model.sort_values([id_col, date_col])

    # Binary outage variable
    if saidi_threshold > 0:
        df_model['outage'] = (df_model[outcome_col] >= saidi_threshold).astype(int)
    else:
        df_model['outage'] = (df_model[outcome_col] > saidi_threshold).astype(int)

    # Default base cols
    if base_cols is None:
        base_cols = ['T', 'P', 'L', 'W']

    df_model[base_cols] = df_model[base_cols].astype(int)

    # Interactions
    df_model['PW']  = df_model['P'] * df_model['W']
    df_model['PL']  = df_model['P'] * df_model['L']
    df_model['WL']  = df_model['W'] * df_model['L']
    df_model['PWL'] = df_model['P'] * df_model['W'] * df_model['L']

    interaction_cols = ['PW', 'PL', 'WL', 'PWL']
    all_cols = base_cols + interaction_cols

    results = []

    def fit_logit(X, y, groups=None):
        try:
            if use_clustered_se:
                if groups is None:
                    raise ValueError("groups must be provided when use_clustered_se=True")

                model = sm.Logit(y, X).fit(
                    disp=0,
                    cov_type='cluster',
                    cov_kwds={'groups': groups}
                )
            else:
                model = sm.Logit(y, X).fit(disp=0)

            issues = []

            if not model.mle_retvals['converged']:
                issues.append('non_converged')

            if np.any(np.abs(model.params) > coef_threshold):
                issues.append('large_coef')

            if not np.isfinite(model.llf):
                issues.append('invalid_loglik')

            return model, issues

        except Exception:
            return None, ['fit_failed']

    for lag in lags:
        temp_df = df_model.copy()
        ew_cols = []

        # Build lagged EW variables WITHIN each EA
        for col in all_cols:
            ew_col = f'EW_{col}_{lag}day'

            if lag == 0:
                temp_df[ew_col] = temp_df[col]
            else:
                temp_df[ew_col] = (
                    temp_df
                    .groupby(id_col)[col]
                    .rolling(window=lag + 1, min_periods=1)
                    .max()
                    .reset_index(level=0, drop=True)
                    .astype(int)
                )

            ew_cols.append(ew_col)

        # Drop zero-variance columns
        valid_ew_cols = [c for c in ew_cols if temp_df[c].nunique() > 1]
        dropped_zero = [c for c in ew_cols if c not in valid_ew_cols]

        # Build controls
        if control_months:
            month_dummies = pd.get_dummies(
                temp_df[date_col].dt.month,
                prefix='month',
                drop_first=True
            ).astype(int)
            month_dummies.index = temp_df.index
            X_full = pd.concat([temp_df[valid_ew_cols], month_dummies], axis=1)
        else:
            month_dummies = None
            X_full = temp_df[valid_ew_cols]

        X_full = sm.add_constant(X_full)
        y = temp_df['outage']

        # No variation in outcome
        if y.nunique() < 2:
            results.append({
                'lag': lag,
                'status': 'no_variation',
                'initial_issues': [],
                'culprit_vars': [],
                'dropped_zero': dropped_zero,
                'used_vars': [],
                'stable_vars': None,
                'use_clustered_se': use_clustered_se
            })
            continue

        # Full model fit
        model, issues = fit_logit(
            X_full,
            y,
            groups=temp_df[cluster_col] if use_clustered_se else None
        )

        if not issues:
            results.append({
                'lag': lag,
                'status': 'ok',
                'initial_issues': [],
                'culprit_vars': [],
                'dropped_zero': dropped_zero,
                'used_vars': valid_ew_cols,
                'stable_vars': valid_ew_cols.copy(),
                'use_clustered_se': use_clustered_se
            })
            continue

        # Test individual variables
        culprit_vars = []
        for var in valid_ew_cols:
            X_test = sm.add_constant(temp_df[[var]])
            _, var_issues = fit_logit(
                X_test,
                y,
                groups=temp_df[cluster_col] if use_clustered_se else None
            )
            if var_issues:
                culprit_vars.append(var)

        # Stepwise drop variables until stable
        stable_vars = None
        for k in range(1, len(valid_ew_cols) + 1):
            for combo in combinations(valid_ew_cols, k):
                remaining = [v for v in valid_ew_cols if v not in combo]

                if not remaining:
                    continue

                if control_months:
                    X_try = pd.concat([temp_df[remaining], month_dummies], axis=1)
                else:
                    X_try = temp_df[remaining]

                X_try = sm.add_constant(X_try)

                _, try_issues = fit_logit(
                    X_try,
                    y,
                    groups=temp_df[cluster_col] if use_clustered_se else None
                )

                if not try_issues:
                    stable_vars = remaining
                    break

            if stable_vars is not None:
                break

        results.append({
            'lag': lag,
            'status': 'unstable',
            'initial_issues': issues,
            'culprit_vars': culprit_vars,
            'dropped_zero': dropped_zero,
            'used_vars': valid_ew_cols,
            'stable_vars': stable_vars,
            'use_clustered_se': use_clustered_se
        })

    return pd.DataFrame(results)

In [50]:
# use_clustered_se=True tells the model that all data is 'not entirely independent' and rows within the same EA may be correlated

In [51]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning


def run_ew_logit_overall(
    df,
    diagnostics_df,
    lags,
    base_cols=None,
    id_col='ea_code9ch',
    date_col='Date',
    outcome_col='SAIDI',
    saidi_threshold=0,
    control_months=False,
    alpha=0.05,
    use_clustered_se=False,
    cluster_col='ea_code9ch'
):
    warnings.filterwarnings("ignore", category=ConvergenceWarning)
    warnings.filterwarnings("ignore", category=RuntimeWarning)

    df_model = df.copy()
    df_model[date_col] = pd.to_datetime(df_model[date_col])
    df_model = df_model.sort_values(date_col)

    # Create binary outage variable
    if saidi_threshold > 0:
        df_model['outage'] = (df_model[outcome_col] >= saidi_threshold).astype(int)
    else:
        df_model['outage'] = (df_model[outcome_col] > saidi_threshold).astype(int)

    # Default base cols
    if base_cols is None:
        base_cols = ['T', 'P', 'L', 'W']

    df_model[base_cols] = df_model[base_cols].astype(int)

    # Construct interactions
    df_model['PW']  = df_model['P'] * df_model['W']
    df_model['PL']  = df_model['P'] * df_model['L']
    df_model['WL']  = df_model['W'] * df_model['L']
    df_model['PWL'] = df_model['P'] * df_model['W'] * df_model['L']

    interaction_cols = ['PW', 'PL', 'WL', 'PWL']
    all_cols = base_cols + interaction_cols

    summary_rows = []
    coef_rows = []

    # Helper: empty summary row for failed/skipped models
    def _empty_row(lag, status, dropped_zero, dropped_EW):
        return {
            'lag':            lag,
            'status':         status,
            'model_type':     'none',
            'n_obs':          np.nan,
            'log_likelihood': np.nan,
            'llr':            np.nan,
            'llr_df':         np.nan,
            'llr_pvalue':     np.nan,
            'mcfadden_r2':    np.nan,
            'significant_EW': [],
            'dropped_zero':   dropped_zero,
            'dropped_EW':     dropped_EW,
            'use_clustered_se': use_clustered_se,
        }

    for lag in lags:
        temp_df = df_model.copy()
        ew_cols = []

        # Build lagged EW variables
        for col in all_cols:
            ew_col = f'EW_{col}_{lag}day'
            if lag == 0:
                temp_df[ew_col] = temp_df[col]
            else:
                temp_df[ew_col] = (
                    temp_df[col]
                    .rolling(window=lag + 1, min_periods=1)
                    .max()
                    .astype(int)
                )
            ew_cols.append(ew_col)

        # Drop zero-variance columns
        valid_ew_cols = [c for c in ew_cols if temp_df[c].nunique() > 1]
        dropped_zero  = [c for c in ew_cols if c not in valid_ew_cols]

        # Build month controls
        if control_months:
            month_dummies = pd.get_dummies(
                temp_df[date_col].dt.month,
                prefix='month',
                drop_first=True
            ).astype(int)
            month_dummies.index = temp_df.index
        else:
            month_dummies = None

        y = temp_df['outage']

        # No variation in outcome
        if y.nunique() < 2:
            summary_rows.append(_empty_row(lag, 'no_variation', dropped_zero, all_cols.copy()))
            continue

        # Pull diagnostics row for this lag
        diag_row = diagnostics_df[diagnostics_df['lag'] == lag]

        if not diag_row.empty:
            diag_row  = diag_row.iloc[0]
            status     = diag_row['status']
            stable_vars = diag_row['stable_vars']
        else:
            status      = 'unknown'
            stable_vars = None

        # Decide model specification
        if status == 'ok':
            model_type = 'full'
            use_vars   = valid_ew_cols

        elif status == 'unstable':
            if isinstance(stable_vars, list) and len(stable_vars) > 0:
                model_type = 'reduced'
                use_vars   = stable_vars
            else:
                model_type = 'none'
                use_vars   = None

        elif status == 'no_variation':
            model_type = 'none'
            use_vars   = None

        else:
            model_type = 'none'
            use_vars   = None

        # Determine dropped EW vars
        if model_type == 'full':
            dropped_EW = dropped_zero
        elif model_type == 'reduced':
            dropped_EW = [v for v in valid_ew_cols if v not in use_vars] + dropped_zero
        else:
            dropped_EW = all_cols.copy()

        # If no usable model
        if model_type == 'none' or use_vars is None or len(use_vars) == 0:
            summary_rows.append(_empty_row(lag, status, dropped_zero, dropped_EW))
            continue

        # Build X
        if control_months:
            X = pd.concat([temp_df[use_vars], month_dummies], axis=1)
        else:
            X = temp_df[use_vars]

        X = sm.add_constant(X)

        # Fit model
        try:
            if use_clustered_se:
                model = sm.Logit(y, X).fit(
                    disp=0,
                    cov_type='cluster',
                    cov_kwds={'groups': temp_df[cluster_col]}
                )
            else:
                model = sm.Logit(y, X).fit(disp=0)

            if not model.mle_retvals['converged']:
                raise ValueError("Non-converged model")

        except Exception:
            summary_rows.append(_empty_row(lag, status, dropped_zero, dropped_EW))
            continue

        # Extract significant EW vars
        sig_ew_vars = []
        for col in all_cols:
            ew_name = f'EW_{col}_{lag}day'
            if ew_name in model.params.index:
                pval = model.pvalues.get(ew_name, np.nan)
                if pd.notna(pval) and pval < alpha:
                    sig_ew_vars.append(col)

        # Store summary row — all model statistics
        summary_rows.append({
            'lag':            lag,
            'status':         status,
            'model_type':     model_type,
            'n_obs':          int(model.nobs),          # N observations
            'log_likelihood': round(model.llf, 3),      # Log-likelihood
            'llr':            round(model.llr, 3),       # LR chi-squared statistic
            'llr_df':         int(model.df_model),       # Degrees of freedom for LR test
            'llr_pvalue':     model.llr_pvalue,          # LR test p-value
            'mcfadden_r2':    round(model.prsquared, 4), # McFadden's pseudo-R²
            'significant_EW': sig_ew_vars,
            'dropped_zero':   dropped_zero,
            'dropped_EW':     dropped_EW,
            'use_clustered_se': use_clustered_se,
        })

        # Confidence intervals
        conf_int = model.conf_int()
        conf_int.columns = ['ci_lower', 'ci_upper']

        # Store coefficient rows
        for var in model.params.index:
            if var == 'const':
                continue

            is_ew = var.startswith('EW_')

            coef_rows.append({
                'lag':        lag,
                'status':     status,
                'model_type': model_type,
                'variable':   var,
                'is_EW':      is_ew,
                'base_var': (
                    var.replace('EW_', '').replace(f'_{lag}day', '')
                    if is_ew else var
                ),
                'coef':       model.params[var],
                'odds_ratio': np.exp(model.params[var]),
                'std_err':    model.bse[var],
                'z_value':    model.tvalues[var],
                'p_value':    model.pvalues[var],
                'ci_lower':   conf_int.loc[var, 'ci_lower'],
                'ci_upper':   conf_int.loc[var, 'ci_upper'],
                'or_ci_lower': np.exp(conf_int.loc[var, 'ci_lower']),
                'or_ci_upper': np.exp(conf_int.loc[var, 'ci_upper']),
                'significant': pd.notna(model.pvalues[var]) and model.pvalues[var] < alpha,
            })

    results_df = pd.DataFrame(summary_rows)
    coef_df    = pd.DataFrame(coef_rows)

    return results_df, coef_df

### Aggregate SAIDI of multiple sites per EA 

In [52]:
aggregated_saidi_df = aggregate_saidi_to_ea(
    combined_saidi_weather,
    ea_col='ea_code9ch',
    site_col='site_id',
    date_col='Date',
    saidi_col='SAIDI',
    saidi_agg= agg_method
)

In [53]:
aggregated_saidi_df.head()

,ea_code9ch,Date,SAIDI,Hot_hour_count,T,Wind_hour_count,W,Precip_sum,P,Lightning_sum,L,PL_day,PWL_day,Month,season,cesi_level,n_sites
0,30200002,2022-01-01,0.0,0,False,0,False,0.000000,False,0.0,False,False,False,1,dry,Low,1
1,30200002,2022-01-02,0.0,0,False,0,False,4.477796,False,0.0,False,False,False,1,dry,Low,1
2,30200002,2022-01-03,0.0,3,False,0,False,0.162920,False,0.0,False,False,False,1,dry,Low,1
3,30200002,2022-01-04,0.0,0,False,0,False,0.041221,False,0.0,False,False,False,1,dry,Low,1
4,30200002,2022-01-05,0.0,0,False,0,False,0.016973,False,0.0,False,False,False,1,dry,Low,1


## *** All Events together -> Full Model 

### Diagnostics df - overall 

In [54]:
sep_df_overall = diagnose_separation_overall(
    aggregated_saidi_df,
    lags=selected_lags,
    saidi_threshold = saidi_threshold,
    # base_cols=['Hot_day','Precip_day','Lightning_day','Wind_day'],
    control_months=True, 
    use_clustered_se=True
)

In [55]:
sep_df_overall

,lag,status,initial_issues,culprit_vars,dropped_zero,used_vars,stable_vars,use_clustered_se
0,0,ok,[],[],[],"[EW_T_0day, EW_P_0day, EW_L_0day, EW_W_0day, E...","[EW_T_0day, EW_P_0day, EW_L_0day, EW_W_0day, E...",True


### Run regression - overall 

In [56]:
results_df_overall, coef_df_overall = run_ew_logit_overall(
    aggregated_saidi_df,
    diagnostics_df = sep_df_overall,  ## pass in diagnostics df 
    lags = selected_lags,
    saidi_threshold = saidi_threshold,
    # base_cols = ['Hot_day', 'Precip_day', 'Lightning_day', 'Wind_day'],
    control_months=True,
    alpha=0.05, 
    use_clustered_se=True
)

In [57]:
results_df_overall

,lag,status,model_type,n_obs,log_likelihood,llr,llr_df,llr_pvalue,mcfadden_r2,significant_EW,dropped_zero,dropped_EW,use_clustered_se
0,0,ok,full,131167,-66701.366,2090.243,19,0.0,0.0154,"[T, P, L, W, WL]",[],[],True


In [58]:
coef_df_overall

,lag,status,model_type,variable,is_EW,base_var,coef,odds_ratio,std_err,z_value,p_value,ci_lower,ci_upper,or_ci_lower,or_ci_upper,significant
0,0,ok,full,EW_T_0day,True,T,0.129827,1.138631,0.034230,3.792813,1.489503e-04,0.062738,0.196915,1.064748,1.217641,True
1,0,ok,full,EW_P_0day,True,P,0.308335,1.361157,0.040674,7.580549,3.440962e-14,0.228614,0.388055,1.256857,1.474111,True
2,0,ok,full,EW_L_0day,True,L,0.411421,1.508961,0.051963,7.917519,2.422971e-15,0.309575,0.513268,1.362846,1.670742,True
3,0,ok,full,EW_W_0day,True,W,-0.107451,0.898120,0.044093,-2.436922,1.481286e-02,-0.193872,-0.021031,0.823763,0.979189,True
4,0,ok,full,EW_PW_0day,True,PW,0.185385,1.203681,0.200550,0.924379,3.552890e-01,-0.207687,0.578456,0.812461,1.783283,False
5,0,ok,full,EW_PL_0day,True,PL,-0.016976,0.983168,0.064148,-0.264634,7.912914e-01,-0.142703,0.108752,0.867011,1.114886,False
6,0,ok,full,EW_WL_0day,True,WL,0.539815,1.715689,0.156289,3.453959,5.524215e-04,0.233495,0.846135,1.263006,2.330622,True
7,0,ok,full,EW_PWL_0day,True,PWL,-0.280352,0.755518,0.266620,-1.051506,2.930261e-01,-0.802917,0.242213,0.448020,1.274065,False
8,0,ok,full,month_2,False,month_2,0.044606,1.045616,0.042371,1.052763,2.924498e-01,-0.038439,0.127651,0.962291,1.136157,False
9,0,ok,full,month_3,False,month_3,0.277393,1.319685,0.037015,7.494085,6.676258e-14,0.204845,0.349941,1.227335,1.418984,True


## Using all EAs -> including interaction terms in full model 

### Interaction terms for T, P, L, W, WL 

In [59]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from itertools import combinations
import warnings
from statsmodels.tools.sm_exceptions import ConvergenceWarning


def diagnose_separation_cesi(
    df,
    lags,
    base_cols=None,
    id_col='ea_code9ch',
    date_col='Date',
    outcome_col='SAIDI',
    saidi_threshold=0,
    control_months=True,
    coef_threshold=10,
    use_clustered_se=False,
    cluster_col='ea_code9ch',
    cesi_col='cesi_level',
    cesi_ref='Low'
):
    df_model = df.copy()
    df_model[date_col] = pd.to_datetime(df_model[date_col])
    df_model = df_model.sort_values([id_col, date_col])

    # Binary outage variable
    if saidi_threshold > 0:
        df_model['outage'] = (df_model[outcome_col] >= saidi_threshold).astype(int)
    else:
        df_model['outage'] = (df_model[outcome_col] > saidi_threshold).astype(int)

    # Base cols: T, P, L, W
    if base_cols is None:
        base_cols = ['T', 'P', 'L', 'W']

    df_model[base_cols] = df_model[base_cols].astype(int)

    # WL interaction
    df_model['WL'] = df_model['W'] * df_model['L']
    weather_cols   = base_cols + ['WL']

    # cesi dummy
    df_model['cesi_High'] = (df_model[cesi_col] != cesi_ref).astype(int)

    results = []

    def fit_logit(X, y, groups=None):
        try:
            if use_clustered_se:
                if groups is None:
                    raise ValueError("groups must be provided when use_clustered_se=True")
                model = sm.Logit(y, X).fit(
                    disp=0,
                    cov_type='cluster',
                    cov_kwds={'groups': groups}
                )
            else:
                model = sm.Logit(y, X).fit(disp=0)

            issues = []
            if not model.mle_retvals['converged']:
                issues.append('non_converged')
            if np.any(np.abs(model.params) > coef_threshold):
                issues.append('large_coef')
            if not np.isfinite(model.llf):
                issues.append('invalid_loglik')

            return model, issues

        except Exception:
            return None, ['fit_failed']

    for lag in lags:
        temp_df = df_model.copy()
        ew_cols = []

        # Build lagged EW variables within each EA
        for col in weather_cols:
            ew_col = f'EW_{col}_{lag}day'
            if lag == 0:
                temp_df[ew_col] = temp_df[col]
            else:
                temp_df[ew_col] = (
                    temp_df
                    .groupby(id_col)[col]
                    .rolling(window=lag + 1, min_periods=1)
                    .max()
                    .reset_index(level=0, drop=True)
                    .astype(int)
                )
            ew_cols.append(ew_col)

        # Build cesi interaction terms
        cesi_int_cols = []
        for ew_col in ew_cols:
            int_col = f'{ew_col}_x_High'
            temp_df[int_col] = temp_df[ew_col] * temp_df['cesi_High']
            cesi_int_cols.append(int_col)

        # All model variables
        all_model_vars = ew_cols + ['cesi_High'] + cesi_int_cols

        # Drop zero-variance columns
        valid_vars   = [c for c in all_model_vars if temp_df[c].nunique() > 1]
        dropped_zero = [c for c in all_model_vars if c not in valid_vars]

        # Build controls
        if control_months:
            month_dummies = pd.get_dummies(
                temp_df[date_col].dt.month,
                prefix='month',
                drop_first=True
            ).astype(int)
            month_dummies.index = temp_df.index
            X_full = pd.concat([temp_df[valid_vars], month_dummies], axis=1)
        else:
            month_dummies = None
            X_full = temp_df[valid_vars]

        X_full = sm.add_constant(X_full)
        y = temp_df['outage']

        # No variation in outcome
        if y.nunique() < 2:
            results.append({
                'lag':              lag,
                'status':           'no_variation',
                'initial_issues':   [],
                'culprit_vars':     [],
                'dropped_zero':     dropped_zero,
                'used_vars':        [],
                'stable_vars':      None,
                'use_clustered_se': use_clustered_se
            })
            continue

        # Full model fit
        model, issues = fit_logit(
            X_full, y,
            groups=temp_df[cluster_col] if use_clustered_se else None
        )

        if not issues:
            results.append({
                'lag':              lag,
                'status':           'ok',
                'initial_issues':   [],
                'culprit_vars':     [],
                'dropped_zero':     dropped_zero,
                'used_vars':        valid_vars,
                'stable_vars':      valid_vars.copy(),
                'use_clustered_se': use_clustered_se
            })
            continue

        # Test individual variables
        culprit_vars = []
        for var in valid_vars:
            X_test = sm.add_constant(temp_df[[var]])
            _, var_issues = fit_logit(
                X_test, y,
                groups=temp_df[cluster_col] if use_clustered_se else None
            )
            if var_issues:
                culprit_vars.append(var)

        # Stepwise drop variables until stable
        stable_vars = None
        for k in range(1, len(valid_vars) + 1):
            for combo in combinations(valid_vars, k):
                remaining = [v for v in valid_vars if v not in combo]
                if not remaining:
                    continue
                if control_months:
                    X_try = pd.concat([temp_df[remaining], month_dummies], axis=1)
                else:
                    X_try = temp_df[remaining]
                X_try = sm.add_constant(X_try)
                _, try_issues = fit_logit(
                    X_try, y,
                    groups=temp_df[cluster_col] if use_clustered_se else None
                )
                if not try_issues:
                    stable_vars = remaining
                    break
            if stable_vars is not None:
                break

        results.append({
            'lag':              lag,
            'status':           'unstable',
            'initial_issues':   issues,
            'culprit_vars':     culprit_vars,
            'dropped_zero':     dropped_zero,
            'used_vars':        valid_vars,
            'stable_vars':      stable_vars,
            'use_clustered_se': use_clustered_se
        })

    return pd.DataFrame(results)

In [60]:
def run_ew_logit_cesi(
    df,
    diagnostics_df,
    lags,
    base_cols=None,
    id_col='ea_code9ch',
    date_col='Date',
    outcome_col='SAIDI',
    saidi_threshold=0,
    control_months=True,
    alpha=0.05,
    use_clustered_se=False,
    cluster_col='ea_code9ch',
    cesi_col='cesi_level',
    cesi_ref='Low'
):
    warnings.filterwarnings("ignore", category=ConvergenceWarning)
    warnings.filterwarnings("ignore", category=RuntimeWarning)

    df_model = df.copy()
    df_model[date_col] = pd.to_datetime(df_model[date_col])
    df_model = df_model.sort_values([id_col, date_col])

    # Binary outage variable
    if saidi_threshold > 0:
        df_model['outage'] = (df_model[outcome_col] >= saidi_threshold).astype(int)
    else:
        df_model['outage'] = (df_model[outcome_col] > saidi_threshold).astype(int)

    # Base cols: T, P, L, W
    if base_cols is None:
        base_cols = ['T', 'P', 'L', 'W']

    df_model[base_cols] = df_model[base_cols].astype(int)

    # WL interaction
    df_model['WL'] = df_model['W'] * df_model['L']
    weather_cols   = base_cols + ['WL']

    # cesi dummy
    df_model['cesi_High'] = (df_model[cesi_col] != cesi_ref).astype(int)

    summary_rows = []
    coef_rows    = []

    # Helper: empty summary row for failed/skipped models
    def _empty_row(lag, status, dropped_zero, dropped_EW):
        return {
            'lag':              lag,
            'status':           status,
            'model_type':       'none',
            'n_obs':            np.nan,
            'log_likelihood':   np.nan,
            'llr':              np.nan,
            'llr_df':           np.nan,
            'llr_pvalue':       np.nan,
            'mcfadden_r2':      np.nan,
            'significant_EW':   [],
            'dropped_zero':     dropped_zero,
            'dropped_EW':       dropped_EW,
            'use_clustered_se': use_clustered_se,
        }

    for lag in lags:
        temp_df = df_model.copy()
        ew_cols = []

        # Build lagged EW variables within each EA
        for col in weather_cols:
            ew_col = f'EW_{col}_{lag}day'
            if lag == 0:
                temp_df[ew_col] = temp_df[col]
            else:
                temp_df[ew_col] = (
                    temp_df
                    .groupby(id_col)[col]
                    .rolling(window=lag + 1, min_periods=1)
                    .max()
                    .reset_index(level=0, drop=True)
                    .astype(int)
                )
            ew_cols.append(ew_col)

        # Build cesi interaction terms
        cesi_int_cols = []
        for ew_col in ew_cols:
            int_col = f'{ew_col}_x_High'
            temp_df[int_col] = temp_df[ew_col] * temp_df['cesi_High']
            cesi_int_cols.append(int_col)

        # All model variables
        all_model_vars = ew_cols + ['cesi_High'] + cesi_int_cols

        # Drop zero-variance columns
        valid_vars   = [c for c in all_model_vars if temp_df[c].nunique() > 1]
        dropped_zero = [c for c in all_model_vars if c not in valid_vars]

        # Build month controls
        if control_months:
            month_dummies = pd.get_dummies(
                temp_df[date_col].dt.month,
                prefix='month',
                drop_first=True
            ).astype(int)
            month_dummies.index = temp_df.index
        else:
            month_dummies = None

        y = temp_df['outage']

        # No variation in outcome
        if y.nunique() < 2:
            summary_rows.append(_empty_row(lag, 'no_variation', dropped_zero, all_model_vars.copy()))
            continue

        # Pull diagnostics row for this lag
        diag_row = diagnostics_df[diagnostics_df['lag'] == lag]

        if not diag_row.empty:
            diag_row    = diag_row.iloc[0]
            status      = diag_row['status']
            stable_vars = diag_row['stable_vars']
        else:
            status      = 'unknown'
            stable_vars = None

        # Decide model specification
        if status == 'ok':
            model_type = 'full'
            use_vars   = valid_vars

        elif status == 'unstable':
            if isinstance(stable_vars, list) and len(stable_vars) > 0:
                model_type = 'reduced'
                use_vars   = stable_vars
            else:
                model_type = 'none'
                use_vars   = None

        elif status == 'no_variation':
            model_type = 'none'
            use_vars   = None

        else:
            model_type = 'none'
            use_vars   = None

        # Determine dropped vars
        if model_type == 'full':
            dropped_EW = dropped_zero

        elif model_type == 'reduced':
            dropped_EW = [v for v in valid_vars if v not in use_vars] + dropped_zero

        else:
            dropped_EW = all_model_vars.copy()

        # If no usable model
        if model_type == 'none' or use_vars is None or len(use_vars) == 0:
            summary_rows.append(_empty_row(lag, status, dropped_zero, dropped_EW))
            continue

        # Build X
        if control_months:
            X = pd.concat([temp_df[use_vars], month_dummies], axis=1)
        else:
            X = temp_df[use_vars]
        X = sm.add_constant(X)

        # Fit model
        try:
            if use_clustered_se:
                model = sm.Logit(y, X).fit(
                    disp=0,
                    cov_type='cluster',
                    cov_kwds={'groups': temp_df[cluster_col]}
                )
            else:
                model = sm.Logit(y, X).fit(disp=0)

            if not model.mle_retvals['converged']:
                raise ValueError("Non-converged model")

        except Exception:
            summary_rows.append(_empty_row(lag, status, dropped_zero, dropped_EW))
            continue

        # Extract significant EW vars (main effects only)
        sig_ew_vars = []
        for col in weather_cols:
            ew_name = f'EW_{col}_{lag}day'
            if ew_name in model.params.index:
                pval = model.pvalues.get(ew_name, np.nan)
                if pd.notna(pval) and pval < alpha:
                    sig_ew_vars.append(col)

        # Store summary row — all model statistics
        summary_rows.append({
            'lag':              lag,
            'status':           status,
            'model_type':       model_type,
            'n_obs':            int(model.nobs),           # N observations
            'log_likelihood':   round(model.llf, 3),       # Log-likelihood
            'llr':              round(model.llr, 3),        # LR chi-squared statistic
            'llr_df':           int(model.df_model),        # Degrees of freedom for LR test
            'llr_pvalue':       model.llr_pvalue,           # LR test p-value
            'mcfadden_r2':      round(model.prsquared, 4),  # McFadden's pseudo-R²
            'significant_EW':   sig_ew_vars,
            'dropped_zero':     dropped_zero,
            'dropped_EW':       dropped_EW,
            'use_clustered_se': use_clustered_se,
        })

        # Confidence intervals
        conf_int = model.conf_int()
        conf_int.columns = ['ci_lower', 'ci_upper']

        # Average Marginal Effects
        try:
            me_frame = model.get_margeff(at='overall').summary_frame()
            me_frame.columns = ['me', 'me_se', 'me_z', 'me_pvalue', 'me_ci_lower', 'me_ci_upper']
        except Exception:
            me_frame = pd.DataFrame()

        # Store coefficient rows
        for var in model.params.index:
            if var == 'const':
                continue

            is_ew   = var.startswith('EW_') and '_x_High' not in var
            is_cesi = 'cesi' in var or '_x_High' in var

            me        = me_frame.loc[var, 'me']        if var in me_frame.index else np.nan
            me_se     = me_frame.loc[var, 'me_se']     if var in me_frame.index else np.nan
            me_pvalue = me_frame.loc[var, 'me_pvalue'] if var in me_frame.index else np.nan

            coef_rows.append({
                'lag':         lag,
                'status':      status,
                'model_type':  model_type,
                'variable':    var,
                'is_EW':       is_ew,
                'is_cesi':     is_cesi,
                'base_var': (
                    var.replace('EW_', '').replace(f'_{lag}day', '')
                    if is_ew else var
                ),
                'coef':        model.params[var],
                'odds_ratio':  np.exp(model.params[var]),
                'std_err':     model.bse[var],
                'z_value':     model.tvalues[var],
                'p_value':     model.pvalues[var],
                'ci_lower':    conf_int.loc[var, 'ci_lower'],
                'ci_upper':    conf_int.loc[var, 'ci_upper'],
                'or_ci_lower': np.exp(conf_int.loc[var, 'ci_lower']),
                'or_ci_upper': np.exp(conf_int.loc[var, 'ci_upper']),
                'significant': pd.notna(model.pvalues[var]) and model.pvalues[var] < alpha,
                'me':          me,
                'me_se':       me_se,
                'me_pvalue':   me_pvalue,
            })

    results_df = pd.DataFrame(summary_rows)
    coef_df    = pd.DataFrame(coef_rows)

    return results_df, coef_df

In [61]:
sep_df_cesi = diagnose_separation_cesi(
    aggregated_saidi_df,
    lags=selected_lags,
    saidi_threshold = saidi_threshold,
    # base_cols=['Hot_day','Precip_day','Lightning_day','Wind_day'],
    control_months=True, 
    use_clustered_se=True
)

In [62]:
sep_df_cesi

,lag,status,initial_issues,culprit_vars,dropped_zero,used_vars,stable_vars,use_clustered_se
0,0,ok,[],[],[],"[EW_T_0day, EW_P_0day, EW_L_0day, EW_W_0day, E...","[EW_T_0day, EW_P_0day, EW_L_0day, EW_W_0day, E...",True


In [63]:
results_df_cesi, coef_df_cesi = run_ew_logit_cesi(
    aggregated_saidi_df,
    diagnostics_df = sep_df_cesi,  ## CESI diagnostics  
    lags = selected_lags,
    saidi_threshold = saidi_threshold,
    # base_cols = ['Hot_day', 'Precip_day', 'Lightning_day', 'Wind_day'],
    control_months=True,
    alpha=0.05, 
    use_clustered_se=True
)

In [64]:
results_df_cesi

,lag,status,model_type,n_obs,log_likelihood,llr,llr_df,llr_pvalue,mcfadden_r2,significant_EW,dropped_zero,dropped_EW,use_clustered_se
0,0,ok,full,131167,-66694.877,2103.223,22,0.0,0.0155,"[T, P, L, W, WL]",[],[],True


In [65]:
coef_df_cesi

,lag,status,model_type,variable,is_EW,is_cesi,base_var,coef,odds_ratio,std_err,z_value,p_value,ci_lower,ci_upper,or_ci_lower,or_ci_upper,significant,me,me_se,me_pvalue
0,0,ok,full,EW_T_0day,True,False,T,0.096183,1.100960,0.037300,2.578606,9.919982e-03,0.023076,0.169290,1.023344,1.184464,True,0.015785,0.006078,9.402303e-03
1,0,ok,full,EW_P_0day,True,False,P,0.312980,1.367494,0.040279,7.770291,7.830598e-15,0.234034,0.391925,1.263688,1.479827,True,0.051365,0.006396,9.718365e-16
2,0,ok,full,EW_L_0day,True,False,L,0.410836,1.508077,0.050034,8.211147,2.190851e-16,0.312771,0.508900,1.367208,1.663461,True,0.067425,0.008489,1.984117e-15
3,0,ok,full,EW_W_0day,True,False,W,-0.095680,0.908754,0.048798,-1.960735,4.990990e-02,-0.191323,-0.000038,0.825866,0.999962,True,-0.015703,0.008039,5.079356e-02
4,0,ok,full,EW_WL_0day,True,False,WL,0.554343,1.740797,0.144093,3.847128,1.195106e-04,0.271926,0.836759,1.312491,2.308873,True,0.090977,0.023658,1.203392e-04
5,0,ok,full,cesi_High,False,True,cesi_High,-0.030655,0.969810,0.088948,-0.344634,7.303693e-01,-0.204990,0.143681,0.814655,1.154516,False,-0.005031,0.014611,7.306019e-01
6,0,ok,full,EW_T_0day_x_High,False,True,EW_T_0day_x_High,0.129638,1.138416,0.078809,1.644961,9.997790e-02,-0.024825,0.284101,0.975480,1.328567,False,0.021276,0.013033,1.025787e-01
7,0,ok,full,EW_P_0day_x_High,False,True,EW_P_0day_x_High,-0.018459,0.981710,0.072423,-0.254880,7.988157e-01,-0.160405,0.123487,0.851799,1.131435,False,-0.003029,0.011880,7.987180e-01
8,0,ok,full,EW_L_0day_x_High,False,True,EW_L_0day_x_High,-0.081237,0.921975,0.126314,-0.643137,5.201350e-01,-0.328809,0.166334,0.719780,1.180968,False,-0.013332,0.020729,5.201005e-01
9,0,ok,full,EW_W_0day_x_High,False,True,EW_W_0day_x_High,-0.021920,0.978318,0.098870,-0.221705,8.245435e-01,-0.215703,0.171862,0.805975,1.187514,False,-0.003597,0.016216,8.244323e-01
